In [ ]:
# %% [markdown]
# # |Twist| & |Bend| Distributions – Two Multimers (Internal Monomers Only)
# 
# This notebook:
# - Takes two directories, each containing monomer_*.csv for one multimer
# - Uses only internal monomers (all monomers except first and last, e.g. 2–7 for an 8-mer)
# - Pools **absolute** twist/bend values across those monomers and timesteps
# - Plots KDE distributions for:
#     1. |Twist| (deg)
#     2. |Bend| (deg)
# - One color and one legend entry per multimer
# - Uses Arial + Illustrator-friendly axis formatting

# %%
%config InlineBackend.figure_formats = ['svg']

# %%
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns

# -----------------------------------------------------------------------------
# GLOBAL STYLE (Illustrator-friendly)
# -----------------------------------------------------------------------------

# Set font to Arial
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']

# Define color scheme for the two multimers
COLORS = {
    'multimer_1': '#1F77B4',   # blue
    'multimer_2': '#FF5733',   # red-orange
}

def format_figure_for_illustrator(ax, show_right_labels=False, show_top_labels=False):
    """
    Full axis box + ticks on all sides, but labels only on left/bottom.
    """

    # 1. Full box (all spines visible)
    for side in ['top', 'right', 'left', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(1.5)
        ax.spines[side].set_color('black')

    # 2. Ticks on all 4 sides
    ax.tick_params(
        which='both',
        direction='out',
        length=4,
        width=2,
        top=True,
        right=True,
        labelsize=10
    )

    # 3. Labels only on left & bottom (unless overridden)
    ax.tick_params(
        axis='x',
        labelbottom=True,
        labeltop=show_top_labels
    )
    ax.tick_params(
        axis='y',
        labelleft=True,
        labelright=show_right_labels
    )

    # 4. Axis label font
    ax.xaxis.label.set_fontsize(10)
    ax.yaxis.label.set_fontsize(10)

    return ax

# -----------------------------------------------------------------------------
# HELPERS: LOAD MULTIMER DATA & SELECT INTERNAL MONOMERS
# -----------------------------------------------------------------------------

def load_multimer_monomers(dir_path):
    """
    Load all monomer_*.csv files from a directory.
    Expected columns in each file: t, twist_deg, bend_deg
    Returns a DataFrame with an added 'monomer' column.
    """
    pattern = os.path.join(dir_path, "monomer_*.csv")
    monomer_files = sorted(glob.glob(pattern))

    if not monomer_files:
        raise FileNotFoundError(f"No 'monomer_*.csv' files found in {dir_path!r}.")

    monomer_pattern = re.compile(r"monomer_(\d+)\.csv")

    monomer_dfs = []
    for f in monomer_files:
        m = monomer_pattern.search(os.path.basename(f))
        if not m:
            continue
        monomer_id = int(m.group(1))
        df = pd.read_csv(f)

        if not {"t", "twist_deg", "bend_deg"}.issubset(df.columns):
            raise ValueError(
                f"File {f} does not have expected columns t, twist_deg, bend_deg."
            )

        df["monomer"] = monomer_id
        monomer_dfs.append(df)

    if not monomer_dfs:
        raise RuntimeError(f"No valid monomer CSVs loaded from {dir_path!r}.")

    monomers = pd.concat(monomer_dfs, ignore_index=True).sort_values(["monomer", "t"])
    return monomers


def get_internal_monomer_ids(df_monomers):
    """
    Return monomer IDs excluding the first and last (by index).
    For an 8-mer with monomers 1..8, this yields 2..7.
    """
    ids = np.sort(df_monomers["monomer"].unique())
    if len(ids) <= 2:
        # Nothing to exclude; return all
        return ids
    return ids[1:-1]


def get_pooled_values(df_monomers, quantity="twist_deg"):
    """
    Pool 'quantity' (twist_deg or bend_deg) across internal monomers
    (all monomers except min and max index) and across time.
    """
    internal_ids = get_internal_monomer_ids(df_monomers)
    df_internal = df_monomers[df_monomers["monomer"].isin(internal_ids)]
    values = df_internal[quantity].dropna().values
    return values, internal_ids

# -----------------------------------------------------------------------------
# PLOTTING FUNCTIONS: TWO MULTIMERS ON ONE PLOT (|quantity|)
# -----------------------------------------------------------------------------

def plot_kde_two_multimers(
    dir_1,
    dir_2,
    quantity="twist_deg",
    label_1="Multimer 1",
    label_2="Multimer 2",
    color_key_1="multimer_1",
    color_key_2="multimer_2",
    x_label="|Twist| (deg)",
    y_label="Probability density",
    title="|Twist| distribution – internal monomers",
    bw_adjust=0.5,
    figsize=(4, 3),
    output_filename=None,
):
    """
    KDE distribution plot for 'quantity' (twist_deg or bend_deg)
    for two multimers (two directories).
    Uses only internal monomers (all except first and last by index).
    Uses the **absolute value** of the quantity before plotting.
    """

    # Load data for each multimer
    mono_1 = load_multimer_monomers(dir_1)
    mono_2 = load_multimer_monomers(dir_2)

    vals_1, internal_ids_1 = get_pooled_values(mono_1, quantity=quantity)
    vals_2, internal_ids_2 = get_pooled_values(mono_2, quantity=quantity)

    # Take absolute value so the sign is ignored
    vals_1 = np.abs(vals_1)
    vals_2 = np.abs(vals_2)

    fig, ax = plt.subplots(figsize=figsize, dpi=300)

    # Multimer 1 KDE
    sns.kdeplot(
        x=vals_1,
        ax=ax,
        fill=False,
        alpha=0.9,
        linewidth=1.5,
        color=COLORS.get(color_key_1, "#1F77B4"),
        bw_adjust=bw_adjust,
        label=f"{label_1}",
    )

    # Multimer 2 KDE
    sns.kdeplot(
        x=vals_2,
        ax=ax,
        fill=False,
        alpha=0.9,
        linewidth=1.5,
        color=COLORS.get(color_key_2, "#FF5733"),
        bw_adjust=bw_adjust,
        label=f"{label_2}",
    )

    # Labels and title
    ax.set_xlabel(x_label, fontsize=10, family="Arial")
    ax.set_ylabel(y_label, fontsize=10, family="Arial")
    ax.set_title(title, fontsize=10, family="Arial", pad=10)

    # Illustrator-friendly formatting
    format_figure_for_illustrator(ax)

    # Legend formatting
    legend = ax.legend(loc="best", frameon=False, fontsize=8)
    for text in legend.get_texts():
        text.set_family("Arial")
        text.set_fontsize(8)

    plt.tight_layout()

    if output_filename is not None:
        plt.savefig(output_filename, format="svg", dpi=300, bbox_inches="tight")
        print(f"Figure saved as: {os.path.abspath(output_filename)}")

    return fig, ax

# %%
# -------------------------------------------------------------------------
# EXAMPLE USAGE: TWO PLOTS (|TWIST| & |BEND|)
# -------------------------------------------------------------------------
# Replace these with your actual directories
dir_multimer_1 = "T3IVT8mer"  # directory containing monomer_*.csv for multimer A
dir_multimer_2 = "K1IVT8mer"  # directory containing monomer_*.csv for multimer B

# ---- Plot 1: |Twist| distribution (internal monomers only) ----
fig_twist, ax_twist = plot_kde_two_multimers(
    dir_1=dir_multimer_1,
    dir_2=dir_multimer_2,
    quantity="twist_deg",
    label_1="T3IVT8mer",
    label_2="K1IVT8mer",
    color_key_1="multimer_1",
    color_key_2="multimer_2",
    x_label="Twist (deg)",
    y_label="Probability density",
    title="Twist distribution (abs)",
    bw_adjust=0.5,
    figsize=(4, 3),
    output_filename="twist_internal_multimers_abs.svg",  # or None if you don't want to save
)

plt.savefig("multi_twist_abs.svg", format='svg', dpi=300, bbox_inches='tight')

# ---- Plot 2: |Bend| distribution (internal monomers only) ----
fig_bend, ax_bend = plot_kde_two_multimers(
    dir_1=dir_multimer_1,
    dir_2=dir_multimer_2,
    quantity="bend_deg",
    label_1="T3IVT8mer",
    label_2="K1IVT8mer",
    color_key_1="multimer_1",
    color_key_2="multimer_2",
    x_label="Curvature (deg)",
    y_label="Probability density",
    title="Curvature distribution",
    bw_adjust=0.5,
    figsize=(4, 3),
    output_filename="bend_internal_multimers_abs.svg",
)

plt.savefig("multi_bend_abs.svg", format='svg', dpi=300, bbox_inches='tight')